# 🔥 Colab B — 3-Layer Neural Net: PyTorch from Scratch
**No built-in layers (no nn.Module, no nn.Linear)**

- ✅ Raw `torch.tensor` with `requires_grad=True`
- ✅ Manual forward pass via `torch.matmul`
- ✅ PyTorch autograd for gradients (no hand-coded backprop)
- ✅ Manual weight update inside `torch.no_grad()`

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
torch.manual_seed(42); np.random.seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '| PyTorch:', torch.__version__)

## 📊 Section 1 — Data (same equation as Colab A)

In [ ]:
N = 1000
x1 = np.random.uniform(-2,2,(N,1)); x2 = np.random.uniform(-2,2,(N,1)); x3 = np.random.uniform(-2,2,(N,1))
y = 2*x1**2 + 3*x2*x3 + np.sin(x1*x2) + 0.5*x3**2 + np.random.normal(0,0.1,(N,1))
X = np.hstack([x1,x2,x3])
X_mean,X_std = X.mean(0),X.std(0); y_mean,y_std = y.mean(),y.std()
X_n = (X-X_mean)/X_std; y_n = (y-y_mean)/y_std
sp = int(0.8*N)
Xtr,Xte = torch.tensor(X_n[:sp],dtype=torch.float32,device=DEVICE), torch.tensor(X_n[sp:],dtype=torch.float32,device=DEVICE)
Ytr,Yte = torch.tensor(y_n[:sp],dtype=torch.float32,device=DEVICE), torch.tensor(y_n[sp:],dtype=torch.float32,device=DEVICE)
print(f'Train: {Xtr.shape}, Test: {Xte.shape}')

## ⚙️ Section 2 — Raw Parameter Tensors (requires_grad=True)

In [ ]:
# He initialization — raw tensors, NO nn.Module
def he(fan_in, fan_out):
    return torch.randn(fan_in, fan_out, device=DEVICE) * (2/fan_in)**0.5

W1 = he(3,64).requires_grad_(True)
b1 = torch.zeros(1,64,device=DEVICE).requires_grad_(True)
W2 = he(64,32).requires_grad_(True)
b2 = torch.zeros(1,32,device=DEVICE).requires_grad_(True)
W3 = he(32,16).requires_grad_(True)
b3 = torch.zeros(1,16,device=DEVICE).requires_grad_(True)
W4 = he(16,1).requires_grad_(True)
b4 = torch.zeros(1,1,device=DEVICE).requires_grad_(True)

all_params = [W1,b1,W2,b2,W3,b3,W4,b4]
total = sum(p.numel() for p in all_params)
print(f'Total parameters: {total}')
for p,n in zip([W1,W2,W3,W4],['W1','W2','W3','W4']):
    print(f'  {n}: {tuple(p.shape)}')

## ➡️ Section 3 — Forward Pass Function

In [ ]:
def forward(X):
    """3-layer forward pass: ReLU, ReLU, Tanh, Linear"""
    A1 = torch.relu(X @ W1 + b1)       # (n,3)→(n,64)
    A2 = torch.relu(A1 @ W2 + b2)      # (n,64)→(n,32)
    A3 = torch.tanh(A2 @ W3 + b3)      # (n,32)→(n,16)
    out = A3 @ W4 + b4                  # (n,16)→(n,1)
    return out

# Quick shape test
with torch.no_grad():
    test_out = forward(Xtr[:4])
    print('Forward output shape:', test_out.shape)

## 🏋️ Section 4 — Training Loop (Manual Adam)

In [ ]:
# Manual Adam state
m_s = [torch.zeros_like(p) for p in all_params]
v_s = [torch.zeros_like(p) for p in all_params]
lr,beta1,beta2,eps = 1e-3, 0.9, 0.999, 1e-8
EPOCHS, BATCH = 1000, 64
history = []

for epoch in range(EPOCHS):
    idx = torch.randperm(Xtr.shape[0])[:BATCH]
    Xb, Yb = Xtr[idx], Ytr[idx]

    # Zero gradients manually
    for p in all_params:
        if p.grad is not None: p.grad.zero_()

    # Forward + Loss
    pred = forward(Xb)
    loss = ((pred - Yb)**2).mean()

    # Autograd backward
    loss.backward()

    # Adam update (no_grad to avoid tracking update operations)
    with torch.no_grad():
        t = epoch + 1
        for i, p in enumerate(all_params):
            g = p.grad
            m_s[i] = beta1*m_s[i] + (1-beta1)*g
            v_s[i] = beta2*v_s[i] + (1-beta2)*(g**2)
            m_hat = m_s[i] / (1 - beta1**t)
            v_hat = v_s[i] / (1 - beta2**t)
            p -= lr * m_hat / (v_hat.sqrt() + eps)

    history.append(loss.item())
    if (epoch+1) % 100 == 0:
        with torch.no_grad():
            vl = ((forward(Xte)-Yte)**2).mean().item()
        print(f'Epoch {epoch+1:4d} | Loss: {loss.item():.5f} | Val: {vl:.5f}')

## 📈 Section 5 — Results

In [ ]:
plt.figure(figsize=(10,4))
plt.semilogy(history,'steelblue',lw=1.5,label='Train MSE')
plt.xlabel('Epoch'); plt.ylabel('MSE (log)'); plt.title('Colab B — PyTorch Scratch')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

with torch.no_grad():
    yp = forward(Xte).cpu().numpy()*y_std+y_mean
    yt = Yte.cpu().numpy()*y_std+y_mean
plt.figure(figsize=(6,6))
plt.scatter(yt,yp,alpha=0.4,s=15,color='steelblue')
mn,mx=yt.min(),yt.max()
plt.plot([mn,mx],[mn,mx],'r--',lw=2,label='Perfect')
plt.xlabel('Actual'); plt.ylabel('Predicted')
plt.title('Colab B — Predicted vs Actual')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f'Test MSE: {((yp-yt)**2).mean():.4f} | MAE: {np.abs(yp-yt).mean():.4f}')